# Ingest Races.csv file
- 0.- Define Schema
- 1.- Read the file using spark dataframe reader API
- 2.- Add Metadata Columns
-     * Source File
-     * Ingestion Timestamp
- 3.- Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers       

In [0]:
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

0.-Define Schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql import functions as F


races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
    
])

Step 1 - Read the CSV file using the dataframe reader API

In [0]:
races_df = (
    spark.read
        .format('csv')
        .option('header', 'True')
        .schema(races_schema)
        #.load('abfss://formula1@eberdatabrickscoursedl.dfs.core.windows.net/landing/races.csv')
        .load(source_file)
                
        
)

In [0]:
display(races_df)

 2.- Add Metadata Columns
-     * Source File
-     * Ingestion Timestamp

In [0]:
races_final_df = add_ingestion_metadata(races_df)
display(races_final_df)


- 3.- Write to bronze delta table

In [0]:
(
    races_final_df
    .write
    .format('delta')
    .mode('overwrite')
    #.saveAsTable('formula1.bronze.races')
    .saveAsTable(table_name)

)

In [0]:
%sql
Select * from formula1.bronze.races

In [0]:
display(spark.table('formula1.bronze.races'))